TF-IDF — From Zero to Search Engine
What you'll learn in this notebook:

Why simple word counting fails for search
What TF (Term Frequency) measures
What IDF (Inverse Document Frequency) measures
How TF × IDF = a powerful relevance score
Build a working search engine from scratch

The Problem: Why Counting Words Fails
Imagine you have 3 documents and someone searches for "pizza recipe".

A naive approach: count how many times each search word appears in each document.

But there's a flaw — the word "the" appears in almost every document. If you count it, documents with lots of "the" will rank higher even if they're completely irrelevant.

TF-IDF solves this by:

✅ Rewarding words that appear often in this specific document (TF)

✅ Penalizing words that appear in every document (IDF)

✅ Multiplying them together to get a final relevance score

In [ ]:
# Our mini document collection — 10 food/cooking articles
# Think of these as web pages returned by a crawler

documents = {
    "S1":  "The best homemade pizza recipe for beginners",
    "S2":  "How to make the best pizza dough from scratch recipe",
    "S3":  "Top 10 tips for baking pizza at home for beginners",
    "S4":  "Easy pasta recipes and Italian cooking techniques",
    "S5":  "A guide to building your own outdoor pizza oven in summer",
    "S6":  "The history of pizza from Naples to New York for foodies",
    "S7":  "Healthy meal prep ideas and cooking tips for temperature control",
    "S8":  "Best wood-fired pizza restaurants in the best cities",
    "S9":  "How to choose the right flour for homemade bread and pizza",
    "S10": "Italian cheese varieties and their uses in traditional cooking",
}

print(f"Total documents: {len(documents)}")

for doc_id,doc_text in documents.items():
    print(f"{doc_id}: {doc_text}")

Total documents: 10
S1: The best homemade pizza recipe for beginners
S2: How to make the best pizza dough from scratch recipe
S3: Top 10 tips for baking pizza at home for beginners
S4: Easy pasta recipes and Italian cooking techniques
S5: A guide to building your own outdoor pizza oven in summer
S6: The history of pizza from Naples to New York for foodies
S7: Healthy meal prep ideas and cooking tips for temperature control
S8: Best wood-fired pizza restaurants in the best cities
S9: How to choose the right flour for homemade bread and pizza
S10: Italian cheese varieties and their uses in traditional cooking


In [ ]:
# Naive approach: just count query word occurrences

query_words = ["best", "pizza", "recipe"]

print("Naive word count scores for query: 'best pizza recipe'")
print(f"{'Doc':<5} {'best':>6} {'pizza':>6} {'recipe':>8} {'TOTAL':>7}")
print("-" * 38)

naive_scores = {}

for doc_id,text in documents.items():
  words = text.lower().split()
  count_best = words.count("best")
  count_pizza = words.count("pizza")
  count_recipe = words.count("recipe")
  total = count_best + count_pizza + count_recipe
  naive_scores[doc_id] = total
  if total > 0:
    print(f"{doc_id:<5} {count_best:>6} {count_pizza:>6} {count_recipe:>8} {total:>7}")

print(naive_scores)
print(sorted(naive_scores.items(), key=lambda x: -x[1])[:5], 1)

print()
print("Ranked by naive count:")
for rank, (doc_id, score) in enumerate(
    sorted(naive_scores.items(), key=lambda x: -x[1])[:5], 1
):
    print(f"  {rank}. {doc_id} (count={score}) — {documents[doc_id]}")

print()
print("⚠️  Problem: S8 ranks #1 only because 'best' appears TWICE.")
print("   But S1 is clearly the most relevant: 'best homemade PIZZA RECIPE for beginners'")

Naive word count scores for query: 'best pizza recipe'
Doc     best  pizza   recipe   TOTAL
--------------------------------------
S1         1      1        1       3
S2         1      1        1       3
S3         0      1        0       1
S5         0      1        0       1
S6         0      1        0       1
S8         2      1        0       3
S9         0      1        0       1
{'S1': 3, 'S2': 3, 'S3': 1, 'S4': 0, 'S5': 1, 'S6': 1, 'S7': 0, 'S8': 3, 'S9': 1, 'S10': 0}
[('S1', 3), ('S2', 3), ('S8', 3), ('S3', 1), ('S5', 1)] 1

Ranked by naive count:
  1. S1 (count=3) — The best homemade pizza recipe for beginners
  2. S2 (count=3) — How to make the best pizza dough from scratch recipe
  3. S8 (count=3) — Best wood-fired pizza restaurants in the best cities
  4. S3 (count=1) — Top 10 tips for baking pizza at home for beginners
  5. S5 (count=1) — A guide to building your own outdoor pizza oven in summer

⚠️  Problem: S8 ranks #1 only because 'best' appears TWICE.
   But S1 is cl

STEP 2 — Text Preprocessing (Getting Documents Ready)
Before calculating TF-IDF, we need to clean the text. Raw text is messy:

"Recipe" and "recipe" should be the same word → lowercase
"the", "a", "and" are noise → remove stop words
"recipes" and "recipe" mean the same thing → stemming
Think of preprocessing as washing vegetables before cooking — you can't skip it.

In [ ]:
documents = {
    "S1":  "The best homemade pizza recipe for beginners",
    "S2":  "How to make the best pizza dough from scratch recipe",
    "S3":  "Top 10 tips for baking pizza at home for beginners",
    "S4":  "Easy pasta recipes and Italian cooking techniques",
    "S5":  "A guide to building your own outdoor pizza oven in summer",
    "S6":  "The history of pizza from Naples to New York for foodies",
    "S7":  "Healthy meal prep ideas and cooking tips for temperature control",
    "S8":  "Best wood-fired pizza restaurants in the best cities",
    "S9":  "How to choose the right flour for homemade bread and pizza",
    "S10": "Italian cheese varieties and their uses in traditional cooking",
}

In [ ]:
# --- Step 2a: Tokenise (split text into individual words) ---

def tokenise(text):
  return text.lower().split()

example = documents["S1"]
tokens = tokenise(example)
print(tokens)

['the', 'best', 'homemade', 'pizza', 'recipe', 'for', 'beginners']


In [ ]:
# --- Step 2b: Remove stop words ---
# Stop words are high-frequency, low-meaning words.
# They appear in EVERY document, so they have zero discriminating power.

STOP_WORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at',
    'to', 'for', 'of', 'is', 'it', 'from', 'how', 'your',
    'their', 'own', 'that', 'this'
}

def remove_stop_words(tokens):
  # 👇 Keep only tokens that are NOT in the stop words set
  return [t for t in tokens if t not in STOP_WORDS]

example = tokenise(documents["S1"])
tokens = remove_stop_words(example)
print(tokens)


['best', 'homemade', 'pizza', 'recipe', 'beginners']


In [ ]:
# --- Step 2c: Stemming ---
# "recipes", "recipe" → same root "recip"
# "cooking", "cooked", "cook" → same root "cook"
#
# We use a manual lookup table here (a real system uses NLTK's PorterStemmer)
# The goal: treat different forms of the same word as ONE word


STEM_RULES = {
    'recipes': 'recip', 'recipe': 'recip',
    'baking': 'bake',   'cooking': 'cook',
    'homemade': 'homemad', 'beginners': 'beginn',
    'building': 'build',   'cities': 'citi',
    'varieties': 'varieti', 'ideas': 'idea',
    'techniques': 'techniqu', 'traditional': 'tradit',
    'restaurants': 'restaur', 'healthy': 'healthi',
    'uses': 'use',     'tips': 'tip',
    'choose': 'choos', 'fired': 'fire',
}

def stem(token):
    # 👇 If we have a rule for this word, use the root; otherwise keep as-is
    return STEM_RULES.get(token, token)

# Show stemming in action
demo_words = ['recipes', 'recipe', 'baking', 'cooking', 'beginners', 'techniques']
print("Stemming examples:")
for w in demo_words:
    print(f"  {w:15s} → {stem(w)}")

print()
print("Why this matters: a search for 'recipe' will now match documents")
print("that say 'recipes' — because both map to 'recip'")

Stemming examples:
  recipes         → recip
  recipe          → recip
  baking          → bake
  cooking         → cook
  beginners       → beginn
  techniques      → techniqu

Why this matters: a search for 'recipe' will now match documents
that say 'recipes' — because both map to 'recip'


In [ ]:
# --- Step 2d: Full pipeline — combine all 3 steps ---
def process(text):
  tokens = tokenise(text)
  cleaned = remove_stop_words(tokens)
  stemed = [stem(t) for t in cleaned]
  return stemed

processed = {}
for doc_id,text in documents.items():
  processed[doc_id] = process(text)

print("Processed documents (these are what TF-IDF works on):")
print()
for doc_id, terms in processed.items():
    print(f"  {doc_id:4s}: {terms}")

Processed documents (these are what TF-IDF works on):

  S1  : ['best', 'homemad', 'pizza', 'recip', 'beginn']
  S2  : ['make', 'best', 'pizza', 'dough', 'scratch', 'recip']
  S3  : ['top', '10', 'tip', 'bake', 'pizza', 'home', 'beginn']
  S4  : ['easy', 'pasta', 'recip', 'italian', 'cook', 'techniqu']
  S5  : ['guide', 'build', 'outdoor', 'pizza', 'oven', 'summer']
  S6  : ['history', 'pizza', 'naples', 'new', 'york', 'foodies']
  S7  : ['healthi', 'meal', 'prep', 'idea', 'cook', 'tip', 'temperature', 'control']
  S8  : ['best', 'wood-fired', 'pizza', 'restaur', 'best', 'citi']
  S9  : ['choos', 'right', 'flour', 'homemad', 'bread', 'pizza']
  S10 : ['italian', 'cheese', 'varieti', 'use', 'tradit', 'cook']
